# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will compare three classifiers that answer the binary question “is this content item labelled as declining?”: Logistic Regression as the readable linear baseline, a shallow Decision Tree for inspectable nonlinear rules, and a constrained Random Forest as a stronger ensemble check. The final choice is based on the same ranking metric used for the Week-4 action queue: Precision@20, Precision@50, and Precision@100 on one untouched test set.

The target is `is_declining_label`, defined as `trend_direction == "down"`. I will not use `trend_direction`, `trend_pct`, IDs, or the `last_30d` / `prev_30d` columns as features. Those fields either define the label, identify rows, or represent the comparison window that produced the label. The model is decision support for review prioritisation, not proof that any feature causes decline.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

# Locate the repository root from either the repo or work/notebooks.
here = Path.cwd()
candidates = [here, *here.parents, Path("/workspaces/Flyrank-ml--internship"), Path("/workspace")]
root_matches = [path for path in candidates if (path / "data" / "raw" / "content_refresh_anonymized.csv").exists()]
if not root_matches:
    local_data_matches = Path.home().glob("Downloads/**/data/raw/content_refresh_anonymized.csv")
    root_matches = [data_path.parents[2] for data_path in local_data_matches]
if not root_matches:
    raise FileNotFoundError(f"Could not locate the repository data from notebook cwd: {here}")
ROOT = root_matches[0]
DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
raw = pd.read_csv(DATA_PATH)

# Build the target and only pre-label, observable feature columns.
raw["is_declining_label"] = raw["trend_direction"].fillna("").str.lower().eq("down").astype(int)
for column in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    raw[column] = pd.to_numeric(raw[column], errors="coerce").fillna(0)
raw["log_impressions_90d"] = np.log1p(raw["impressions_90d"])
raw["log_clicks_90d"] = np.log1p(raw["clicks_90d"])
raw["log_sessions_90d"] = np.log1p(raw["sessions_90d"])
raw["log_ai_sessions_90d"] = np.log1p(raw["ai_sessions_90d"])

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
numeric_features = [column for column in numeric_features if column in raw.columns]
categorical_features = [column for column in categorical_features if column in raw.columns]
feature_columns = numeric_features + categorical_features

# Reproduce the Week-4 baseline score on this same frame.
def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    if values.max() == values.min():
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - values.min()) / (values.max() - values.min())

raw["visibility_score"] = percentile_rank(np.log1p(raw["impressions_90d"]))
raw["freshness_risk_score"] = percentile_rank(raw["days_since_last_update"])
raw["position_opportunity_score"] = (1 - normalize(raw["avg_position"].clip(lower=1, upper=50))) * raw["visibility_score"] * (raw["avg_position"] > 0).astype(int)
raw["depth_gap_score"] = (1 - percentile_rank(raw["word_count"])) * raw["visibility_score"]
raw["baseline_score"] = (0.40 * raw["visibility_score"] + 0.30 * raw["freshness_risk_score"] + 0.25 * raw["position_opportunity_score"] + 0.05 * raw["depth_gap_score"]).clip(0, 1)

assert raw["is_declining_label"].nunique() == 2
assert not set(["trend_direction", "trend_pct", "content_id", "client_id"]).intersection(feature_columns)
print(f"Rows: {len(raw):,}; clients: {raw['client_id'].nunique():,}; positive rate: {raw['is_declining_label'].mean():.3f}")
print(f"Numeric features: {len(numeric_features)}; categorical features: {len(categorical_features)}")

Rows: 30,000; clients: 32; positive rate: 0.542
Numeric features: 18; categorical features: 8


## 2. Split design

I will hold out complete clients rather than randomly mixing rows from the same client across train and test. This is a stricter check of whether the pattern transfers to an unseen client. Twenty percent of the pseudonymous clients are assigned to test with a fixed seed. If that holdout cannot contain both label classes, the notebook falls back to a stratified row split and reports that weaker design explicitly.

In [2]:
unique_clients = raw["client_id"].fillna("unknown").astype(str).drop_duplicates().to_numpy()
all_indices = np.arange(len(raw))
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = raw["client_id"].fillna("unknown").astype(str).isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]
split_strategy = "client_holdout"

if (
    len(train_indices) == 0
    or len(test_indices) == 0
    or raw.iloc[train_indices]["is_declining_label"].nunique() < 2
    or raw.iloc[test_indices]["is_declining_label"].nunique() < 2
):
    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=raw["is_declining_label"],
    )
    split_strategy = "stratified_row_holdout"

train_frame = raw.iloc[train_indices].copy()
test_frame = raw.iloc[test_indices].copy()
print(f"Split: {split_strategy}")
print(f"Train rows: {len(train_frame):,}; test rows: {len(test_frame):,}")
print(f"Train clients: {train_frame['client_id'].nunique():,}; test clients: {test_frame['client_id'].nunique():,}")
print(f"Train positive rate: {train_frame['is_declining_label'].mean():.3f}; test positive rate: {test_frame['is_declining_label'].mean():.3f}")
assert set(train_frame["content_id"]).isdisjoint(set(test_frame["content_id"]))

Split: client_holdout
Train rows: 27,675; test rows: 2,325
Train clients: 26; test clients: 6
Train positive rate: 0.555; test positive rate: 0.391


## 3. Train + compare vs my baseline

The baseline and every model are evaluated on `test_indices` using the same positive label and the same ranking metrics. The baseline score is computed without training. The learned models return probabilities, because the task is to rank pages for review rather than only emit a yes/no class. The model selection rule is highest Precision@50, with average precision and ROC AUC used only as tie-breakers.

In [3]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

models = {
    "logistic_regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]
    ),
    "decision_tree": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE)),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(class_weight="balanced_subsample", n_estimators=100, max_depth=10, min_samples_leaf=25, n_jobs=-1, random_state=RANDOM_STATE)),
        ]
    ),
}

X_train = train_frame[feature_columns]
X_test = test_frame[feature_columns]
y_train = train_frame["is_declining_label"].astype(int)
y_test = test_frame["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k):
    scored = pd.DataFrame({"label": np.asarray(y_true), "score": np.asarray(scores)})
    return float(scored.sort_values("score", ascending=False).head(min(k, len(scored)))["label"].mean())

def metrics_for(y_true, scores):
    binary = (np.asarray(scores) >= 0.5).astype(int)
    return {
        "precision@20": precision_at_k(y_true, scores, 20),
        "precision@50": precision_at_k(y_true, scores, 50),
        "precision@100": precision_at_k(y_true, scores, 100),
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "threshold_precision": float(precision_score(y_true, binary, zero_division=0)),
    }

results = {"baseline": metrics_for(y_test, test_frame["baseline_score"].to_numpy())}
probabilities = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    probabilities[model_name] = model.predict_proba(X_test)[:, 1]
    results[model_name] = metrics_for(y_test, probabilities[model_name])

comparison = pd.DataFrame(results).T.sort_values(
    ["precision@50", "average_precision", "roc_auc"], ascending=False
)
print(comparison.round(3).to_string())
best_model_name = comparison.drop(index="baseline").index[0]
best_model = models[best_model_name]
print(f"\nSelected model: {best_model_name} by Precision@50")
assert len(comparison) == 4
assert comparison.index.is_unique
assert comparison.loc["baseline", "precision@50"] >= 0
assert comparison.loc[best_model_name, "precision@50"] >= 0

output_dir = ROOT / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output_dir / "w05_model_comparison.csv")

                     precision@20  precision@50  precision@100  average_precision  roc_auc  threshold_precision
random_forest                0.80          0.66           0.71              0.610    0.750                0.569
decision_tree                0.55          0.62           0.60              0.575    0.742                0.569
logistic_regression          0.35          0.40           0.43              0.529    0.708                0.575
baseline                     0.15          0.24           0.36              0.468    0.627                0.499

Selected model: random_forest by Precision@50


## 4. Errors and interpretation

I will inspect both kinds of threshold errors and the strongest model features. False positives are pages the model prioritises that are not labelled declining; false negatives are labelled declining pages that receive a lower probability. These examples show where the ranking is uncertain and keep the metric connected to real rows.

In [4]:
best_scores = probabilities[best_model_name]
error_frame = test_frame[["content_id", "client_id", "content_type", "is_declining_label", "impressions_90d", "clicks_90d", "days_since_last_update", "avg_position"]].copy()
error_frame["predicted_probability"] = best_scores
error_frame["predicted_label"] = (best_scores >= 0.5).astype(int)
false_positives = error_frame[(error_frame["predicted_label"] == 1) & (error_frame["is_declining_label"] == 0)].sort_values("predicted_probability", ascending=False)
false_negatives = error_frame[(error_frame["predicted_label"] == 0) & (error_frame["is_declining_label"] == 1)].sort_values("predicted_probability", ascending=True)

print("Top false positives: high predicted decline probability but observed label is 0")
print(false_positives.head(3).to_string(index=False))
print("\nTop false negatives: observed label is 1 but predicted probability is low")
print(false_negatives.head(3).to_string(index=False))

model_step = best_model.named_steps["model"]
preprocessor_step = best_model.named_steps["preprocessor"]
transformed_names = preprocessor_step.get_feature_names_out()
if hasattr(model_step, "coef_"):
    importance_values = np.abs(model_step.coef_[0])
elif hasattr(model_step, "feature_importances_"):
    importance_values = model_step.feature_importances_
else:
    importance_values = np.zeros(len(transformed_names))
importance = pd.DataFrame({"feature": transformed_names, "importance": importance_values}).sort_values("importance", ascending=False)
print("\nTop 10 model features")
print(importance.head(10).to_string(index=False))

print("\nInterpretation")
print(
    f"The selected {best_model_name} model ranks pages using observable metadata and trailing-window activity. "
    "The error examples show that a high score is a review priority, not a guaranteed diagnosis. "
    "Feature importance is associative: it identifies useful signals for this split, not causes of decline."
)
assert len(false_positives) + len(false_negatives) <= len(error_frame)
assert len(importance) > 0
assert set(error_frame["content_id"]).issubset(set(test_frame["content_id"]))

Top false positives: high predicted decline probability but observed label is 0
          content_id         client_id    content_type  is_declining_label  impressions_90d  clicks_90d  days_since_last_update  avg_position  predicted_probability  predicted_label
content_331182ca4cae client_f74efabef1 keyword article                   0             3026           0                      20          35.9               0.760562                1
content_d2dffcc697a4 client_f74efabef1 keyword article                   0             5091          10                      20          14.1               0.745269                1
content_643f585dc7f7 client_f74efabef1 keyword article                   0              761           3                      20          25.1               0.740013                1

Top false negatives: observed label is 1 but predicted probability is low
          content_id         client_id    content_type  is_declining_label  impressions_90d  clicks_90d  days_since_l

## Self-check

- [x] Method choice names a simple model, a nonlinear model, and the selection metric.
- [x] The split is grouped by client when possible and uses a fixed seed.
- [x] Baseline and models use the same test rows and Precision@20/50/100 metrics.
- [x] The notebook reports false positives, false negatives, and top features.
- [x] Ran every code cell top to bottom in a clean kernel and inspected the outputs.
- [x] No client names, URLs, private queries, or unsupported causal claims appear in the notebook narrative.
- [ ] Commit the executed notebook under `work/notebooks/` and submit the repository URL.